# OpenEnv SME Negotiator — GRPO Training on Colab

**Colab workflow** for a tiny GRPO run on the SME Liquidity environment.

This notebook is intentionally structured like the kube-sre-gym Colab flow:
Install → Config → Smoke test → Imports/utilities → GRPO setup → Train → Reward curves → Optional upload.

## 1. Install Dependencies

Install project + RL extras and common training dependencies.

In [ ]:
# If needed, clone your repo first:
# !git clone https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o.git
# %cd OpenEnv_SME_Negotiator-2.o

%pip install -q -U pip
%pip install -q -e ".[rl]"
%pip install -q "trl>=0.29.0" "peft" "transformers" "datasets" "huggingface_hub>=0.20.0" "matplotlib"

## 2. Configuration

Set model, task, and tiny training hyperparameters. Add `HF_TOKEN` in Colab Secrets (key icon) if required.

In [ ]:
import os
from pathlib import Path

# --- HuggingFace token from Colab secrets (optional) ---
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

# --- Core config ---
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TASK_NAME = "liquidity-correlation-hard"
TOTAL_PERIODS = 2

# Tiny run defaults for Colab demo
STEPS = 10
NUM_SAMPLES = 8
OUTPUT_DIR = Path("outputs/colab_grpo_sme_liquidity")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print("  MODEL_NAME   :", MODEL_NAME)
print("  TASK_NAME    :", TASK_NAME)
print("  TOTAL_PERIODS:", TOTAL_PERIODS)
print("  STEPS        :", STEPS)
print("  NUM_SAMPLES  :", NUM_SAMPLES)
print("  OUTPUT_DIR   :", OUTPUT_DIR)

## 3. Smoke Test — Verify Environment Connectivity

Run a reset on the in-process SME liquidity environment.

In [ ]:
from server.environment import SMELiquidityEnvironment

env = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
obs = env.reset(seed=42, difficulty="hard", task_name=TASK_NAME)
print("Smoke test OK. Initial observation:")
print(obs)

## 4. Import Training Utilities from Package

Import helper functions used by your RL demo path.

In [ ]:
import matplotlib.pyplot as plt
from rl.demo import run_heuristic_episode, demo_train_grpo, run_policy_episode

print("Imports successful.")

## 5. GRPO Training Setup

Run baseline heuristic episodes before training.

In [ ]:
baseline_runs = [
    run_heuristic_episode(seed=i, total_periods=TOTAL_PERIODS, task_name=TASK_NAME)
    for i in range(5)
]
baseline_rewards = [item["total_reward"] for item in baseline_runs]

print("Baseline rewards:", baseline_rewards)
print("\nSample baseline transcript:\n")
print(baseline_runs[0].get("transcript", "(no transcript field)"))

## 6. Train!

Launch tiny GRPO training demo.

In [ ]:
print("Starting GRPO training...")
print(f"  Model      : {MODEL_NAME}")
print(f"  Task       : {TASK_NAME}")
print(f"  Steps      : {STEPS}")
print(f"  Samples    : {NUM_SAMPLES}")
print(f"  Output dir : {OUTPUT_DIR}")

history = demo_train_grpo(
    model_name=MODEL_NAME,
    steps=STEPS,
    total_periods=TOTAL_PERIODS,
    task_name=TASK_NAME,
    num_samples=NUM_SAMPLES,
    output_dir=str(OUTPUT_DIR),
)

print("Training done.")
history

## 7. Reward Curves

Visualize average reward across training steps.

In [ ]:
steps = history.get("steps", [])
avg_reward = history.get("avg_reward", [])

plt.figure(figsize=(7, 4))
plt.plot(steps, avg_reward, marker="o")
plt.xlabel("Training step")
plt.ylabel("Average episode reward")
plt.title("Tiny GRPO Demo on SMELiquidityEnvironment")
plt.grid(alpha=0.3)
plt.show()

## 8. Push to Hub (Optional)

Optional: evaluate before/after and keep checkpoint path for later upload scripts.

In [ ]:
print("=== Before Training (Heuristic) ===")
print(run_policy_episode(policy="heuristic", seed=123, total_periods=TOTAL_PERIODS, task_name=TASK_NAME))

print("\n=== After Tiny GRPO Demo ===")
print(
    run_policy_episode(
        policy="trained",
        seed=123,
        total_periods=TOTAL_PERIODS,
        task_name=TASK_NAME,
        checkpoint_path=history.get("checkpoint_path"),
    )
)

print("\nCheckpoint path:", history.get("checkpoint_path"))
# If you have a push helper in your repo, call it here.